# Séance optionnelle · Software engineering · ⭐

**Niveau : ⭐ Débutant**

**Séance optionnelle.** Elle ne fait pas partie du parcours obligatoire : elle sert à comprendre *le décor* dans lequel tout le reste se passe — la machine, la mémoire, et l'endroit où le code tourne.

Trois questions que tout le monde se pose un jour, et auxquelles personne ne répond jamais clairement :

1. Tout le monde dit que Python est **lent**. C'est vrai ? Et alors, on fait comment ?
2. Combien de **mémoire** prennent mes données, et que faire quand le fichier est plus gros que ma machine ?
3. Je fais tourner ça **où** — mon ordinateur, Colab, le cloud — et comment quelqu'un d'autre relance-t-il mon projet à l'identique ?

On n'y répond pas par des affirmations, mais en **mesurant** : un chronomètre, un compteur de mémoire, un fichier de dépendances.

Tout tourne dans **Google Colab**, rien à installer, aucun compte payant, aucune clé. Exécute chaque cellule avec `Maj + Entrée`.

**Livrable de la séance** : le `requirements.txt` de ton propre projet, et ta fiche « où faire tourner mon projet » avec ta décision et ce qui la justifie.


## 1. Pourquoi Python est lent, et comment on le contourne

Oui, Python est lent. Non, ce n'est pas un problème. Voici pourquoi.

**Compilé** (Rust, C++) : ton code est traduit **une fois pour toutes** en instructions machine, avant d'être lancé. C'est un livre déjà traduit : à la lecture, plus rien à faire.

**Interprété** (Python) : ton code est lu et traduit **ligne par ligne, à chaque exécution**. C'est un interprète en direct : à chaque phrase, il retraduit. Souple, pratique, mais il y a un coût à chaque tour de boucle.

Et il y a pire : dans Python, `3` n'est pas « le nombre 3 sur 8 octets ». C'est un **objet** avec un type, un compteur de références et une valeur — une trentaine d'octets, quelque part en mémoire. Une boucle sur 5 millions de nombres, c'est 5 millions d'allers-retours vers 5 millions de petits objets éparpillés.

Mesurons. Le mot d'ordre de cette séance : on ne croit pas, on chronomètre.

In [ ]:
import time
import numpy as np

N = 5_000_000     # cinq millions de valeurs

valeurs = [float(i) for i in range(N)]   # une liste Python : 5 millions de petits objets
tableau = np.arange(N, dtype=float)      # un tableau NumPy : 5 millions de nombres collés en mémoire

print("liste Python  :", len(valeurs), "valeurs")
print("tableau NumPy :", tableau.size, "valeurs,", tableau.nbytes / 1e6, "Mo d'un seul tenant")

In [ ]:
# Version 1 : la boucle Python. On applique le même petit calcul à chaque valeur.
depart = time.perf_counter()

total_boucle = 0.0
for v in valeurs:
    total_boucle += v ** 0.5 * 2 + 1

duree_boucle = time.perf_counter() - depart
print(f"boucle Python : {duree_boucle:.3f} seconde(s)   → total = {total_boucle:.1f}")

In [ ]:
# Version 2 : exactement le même calcul, écrit pour NumPy — sans aucune boucle.
depart = time.perf_counter()

total_numpy = float((np.sqrt(tableau) * 2 + 1).sum())

duree_numpy = time.perf_counter() - depart
print(f"NumPy         : {duree_numpy:.3f} seconde(s)   → total = {total_numpy:.1f}")
print()
print(f"Même résultat : {round(total_boucle) == round(total_numpy)}")
print(f"NumPy est ×{duree_boucle / duree_numpy:.0f} plus rapide")

### Pourquoi un tel écart

Le calcul est le même. Ce qui change, c'est **qui fait la boucle**.

| | Boucle Python | NumPy |
|---|---|---|
| Qui exécute la boucle | L'interpréteur Python, tour par tour | Du code **C déjà compilé**, à l'intérieur de NumPy |
| Où sont les données | 5 millions d'objets éparpillés en mémoire | Un seul bloc de 40 Mo, valeurs collées les unes aux autres |
| Vérifications | À chaque tour : « c'est bien un nombre ? » | Une seule fois, au départ |
| Processeur | Une valeur à la fois | Plusieurs valeurs par instruction (SIMD) |

Le point important : **NumPy et pandas ne sont pas écrits en Python.** Leur cœur est écrit en C (et de plus en plus en Rust). Quand tu écris `np.sqrt(tableau)`, tu écris une ligne de Python qui déclenche une boucle en C.

Autrement dit : Python est lent, mais **on ne s'en sert pas pour calculer**. On s'en sert pour *piloter* du code rapide. C'est un langage de commande, pas un langage de calcul — et c'est exactement pour ça qu'il a gagné.

Le nom de cette technique : la **vectorisation**. La règle pratique :

> Si tu écris une boucle `for` sur les lignes d'un DataFrame, il existe presque toujours une façon 50 fois plus rapide de l'écrire.

In [ ]:
import pandas as pd

df = pd.DataFrame({"prix": np.arange(1, 300_001, dtype=float)})

# Version lente : .apply() appelle une fonction Python, ligne par ligne
depart = time.perf_counter()
df["ttc_lent"] = df["prix"].apply(lambda p: p * 1.2)
duree_apply = time.perf_counter() - depart

# Version vectorisée : une seule opération sur toute la colonne
depart = time.perf_counter()
df["ttc_rapide"] = df["prix"] * 1.2
duree_vect = time.perf_counter() - depart

print(f".apply()    : {duree_apply:.4f} s")
print(f"vectorisé   : {duree_vect:.4f} s")
print(f"→ ×{duree_apply / duree_vect:.0f} plus rapide, pour un résultat identique :",
      bool((df["ttc_lent"] == df["ttc_rapide"]).all()))

### À toi · Vectorise cette boucle

La cellule ci-dessous calcule une remise ligne par ligne, avec une boucle. Réécris le même calcul **sans boucle**, dans `remises_vect`, et chronomètre-le.

Règle métier : la remise vaut `prix * 0.3` si le prix dépasse 100, et `prix * 0.1` sinon.

Résultat attendu : les deux listes de remises sont identiques, et la version vectorisée est nettement plus rapide.

<details><summary>Indice</summary>

`np.where(condition, valeur_si_vrai, valeur_si_faux)` applique un `if / else` à tout un tableau d'un seul coup.
Avec pandas, `df["prix"] > 100` te donne déjà une colonne de `True` / `False`.

</details>

In [ ]:
# À toi
prix = df["prix"].to_numpy()

# --- version lente, déjà écrite ---
depart = time.perf_counter()
remises_boucle = []
for p in prix:
    remises_boucle.append(p * 0.3 if p > 100 else p * 0.1)
remises_boucle = np.array(remises_boucle)
duree_lente = time.perf_counter() - depart

# --- version vectorisée, à toi ---
depart = time.perf_counter()
remises_vect = None          # ← une seule ligne, sans boucle
duree_rapide = time.perf_counter() - depart

print(f"boucle    : {duree_lente:.4f} s")
print(f"vectorisé : {duree_rapide:.4f} s")

<details><summary>Solution</summary>

```python
remises_vect = np.where(prix > 100, prix * 0.3, prix * 0.1)

print("identiques :", bool(np.array_equal(remises_boucle, remises_vect)))
print(f"→ ×{duree_lente / max(duree_rapide, 1e-9):.0f} plus rapide")
```

`np.where` lit les 300 000 valeurs en une seule instruction compilée. La boucle, elle, fait 300 000 allers-retours dans l'interpréteur *et* 300 000 `append` dans une liste qui se réagrandit au fur et à mesure.

</details>

### Et si NumPy ne suffit pas : Polars

Depuis quelques années, un nouvel outil monte : **[Polars](https://pola.rs)**. C'est un remplaçant de pandas, écrit en **Rust**, qui s'utilise depuis Python avec une syntaxe très proche :

```python
import polars as pl
df = pl.read_csv("ventes.csv")
df.filter(pl.col("prix") > 100).group_by("ville").agg(pl.col("prix").mean())
```

Il est souvent 5 à 20 fois plus rapide que pandas sur de gros fichiers, parce qu'il utilise tous les cœurs du processeur et ne charge que les colonnes dont il a besoin.

*Simple mention ici : on ne l'installe pas et on ne s'en sert pas dans ce cours.* Retiens juste le principe, qui résume toute cette partie : **la couche visible est en Python, le moteur est en C ou en Rust.** C'est vrai de NumPy, de pandas, de Polars, de PyTorch. Tu écris toujours du Python ; ce n'est jamais Python qui calcule.

## 2. La mémoire, en la mesurant

Deuxième question que personne ne pose avant de se faire avoir : **est-ce que mes données tiennent en mémoire ?**

Petit rappel de vocabulaire :

- Le **disque** (SSD), c'est le placard : grand, lent, garde tout quand on éteint.
- La **RAM** (mémoire vive), c'est le plan de travail : petit, rapide, vidé quand on éteint.

Pour travailler sur des données, il faut les poser sur le plan de travail. Un Colab gratuit t'en donne environ **12 Go**. Ce n'est pas rien, mais un CSV de 3 Go peut occuper 10 Go une fois chargé dans pandas — le format texte est compact, les objets Python ne le sont pas.

Fabriquons des données et regardons ce qu'elles pèsent vraiment.

In [ ]:
import numpy as np
import pandas as pd

generateur = np.random.default_rng(0)      # graine fixe → tout le monde a les mêmes données
n = 300_000

villes = ["Paris", "Lyon", "Marseille", "Lille", "Nantes", "Toulouse", "Nice", "Rennes", "Brest", "Dijon"]
categories = ["sport", "musique", "cinéma", "jeux", "lecture"]

df = pd.DataFrame({
    "ville": generateur.choice(villes, n),
    "categorie": generateur.choice(categories, n),
    "note": generateur.integers(0, 11, n),
    "duree_min": generateur.random(n) * 120,
})

df.info(memory_usage="deep")     # "deep" = compte aussi le vrai poids des textes

### Comment lire ce tableau

Trois choses à repérer :

- **`memory usage: ... MB`** : le poids réel du DataFrame en mémoire. Sans `deep=True`, pandas ne compte que les pointeurs et ment de façon spectaculaire sur les colonnes de texte.
- **`dtype: object`** : c'est le type fourre-tout de pandas pour le texte. Chaque case contient un *pointeur* vers une chaîne Python rangée ailleurs. Coûteux.
- **`dtype: int64` / `float64`** : 8 octets par valeur, collés les uns aux autres. Efficace.

Regarde le détail colonne par colonne : deux colonnes de texte pèsent bien plus lourd que deux colonnes de nombres, alors qu'elles contiennent le même nombre de valeurs.

In [ ]:
def espace(nombre):
    """Affiche 300000 comme « 300 000 » : plus lisible qu'un mur de chiffres."""
    return f"{nombre:,}".replace(",", " ")

poids = df.memory_usage(deep=True) / 1e6
print(poids.round(2).to_string(), "\n")
print(f"Total : {poids.sum():.1f} Mo pour {espace(n)} lignes")

### Le type `category` : quand un texte ne prend que 1 octet

`ville` ne contient que **10 valeurs différentes**, répétées 300 000 fois. Stocker 300 000 fois la chaîne `"Marseille"` est un gâchis.

Le type `category` de pandas range les 10 valeurs dans un petit dictionnaire à part, et remplace la colonne par des **numéros** (0 à 9) tenant sur 1 octet. Les valeurs affichées ne changent pas ; seul le rangement change.

Mesurons le gain.

In [ ]:
avant = df.memory_usage(deep=True).sum() / 1e6

df_leger = df.copy()
df_leger["ville"] = df_leger["ville"].astype("category")

apres = df_leger.memory_usage(deep=True).sum() / 1e6

print(f"avant : {avant:.1f} Mo")
print(f"après : {apres:.1f} Mo")
print(f"gain  : {100 * (avant - apres) / avant:.0f} %  — pour une seule colonne convertie")
print()
print("Les données, elles, n'ont pas bougé :")
print(df_leger["ville"].value_counts().head(3).to_string())

**Quand `category` marche** : peu de valeurs différentes, beaucoup de répétitions. Une ville, un genre de film, un statut, un département, un type de Pokémon.

**Quand ça ne sert à rien, voire nuit** : une colonne où presque chaque valeur est unique (un identifiant, un e-mail, un commentaire libre). Le dictionnaire devient aussi gros que la colonne, et on a payé la conversion pour rien.

La règle qui va bien : convertis si le nombre de valeurs différentes est **inférieur à la moitié** du nombre de lignes. En pratique, on convertit quand c'est inférieur à quelques pourcents.

### À toi · Alléger la deuxième colonne

Convertis **aussi** `categorie` en `category` dans `df_leger`, puis calcule le gain total en pourcentage par rapport à `avant`, dans `gain_total`.

Résultat attendu : un gain d'environ 88 % — les deux colonnes de texte pesaient à elles seules 39 des 44 Mo.

<details><summary>Indice</summary>

Même ligne que ci-dessus, en changeant le nom de la colonne. Pour le gain : `100 * (avant - final) / avant`.

</details>

In [ ]:
# À toi
df_leger["categorie"] = None       # ← convertis la colonne

final = df_leger.memory_usage(deep=True).sum() / 1e6
gain_total = None                  # ← le gain en %, par rapport à `avant`

print(f"{avant:.1f} Mo → {final:.1f} Mo   (gain : {gain_total} %)")

<details><summary>Solution</summary>

```python
df_leger["categorie"] = df["categorie"].astype("category")

final = df_leger.memory_usage(deep=True).sum() / 1e6
gain_total = round(100 * (avant - final) / avant)

print(f"{avant:.1f} Mo → {final:.1f} Mo   (gain : {gain_total} %)")   # ≈ 44 Mo → 5 Mo, gain ≈ 88 %
```

Deux lignes de code, les neuf dixièmes de la mémoire économisés. Sur un fichier de 8 Go qui ne rentrait pas, ça peut suffire à le faire rentrer.

</details>

### Quand le fichier est plus gros que la RAM : `chunksize`

Il arrive un moment où aucune astuce de type ne suffit : le fichier fait 20 Go, la machine a 12 Go. `pd.read_csv()` plante, purement et simplement.

La solution n'est pas une plus grosse machine : c'est de **ne jamais tout charger**. On lit le fichier par **morceaux** (des *chunks*), on traite chaque morceau, on garde seulement le résumé, et on jette le morceau.

C'est exactement comme vider une piscine avec un seau : on ne soulève jamais la piscine.

Écrivons un vrai CSV sur le disque pour le démontrer.

In [ ]:
import os

df.to_csv("gros_fichier.csv", index=False)
taille_mo = os.path.getsize("gros_fichier.csv") / 1e6

print(f"gros_fichier.csv : {taille_mo:.1f} Mo sur le disque, {espace(len(df))} lignes")
print("(sur ta machine il ferait 20 Go ; le principe est identique)")

In [ ]:
# Lecture par morceaux de 50 000 lignes : on ne dépasse jamais 50 000 lignes en mémoire
depart = time.perf_counter()

nb_morceaux = 0
somme_notes = 0
nb_lignes = 0

for morceau in pd.read_csv("gros_fichier.csv", chunksize=50_000):
    nb_morceaux += 1
    somme_notes += morceau["note"].sum()     # on ne garde que le résumé
    nb_lignes += len(morceau)
    # `morceau` est oublié ici : la mémoire est rendue avant le tour suivant

print(f"{nb_morceaux} morceaux lus, {espace(nb_lignes)} lignes au total")
print(f"note moyenne : {somme_notes / nb_lignes:.3f}")
print(f"en {time.perf_counter() - depart:.2f} s, sans jamais charger tout le fichier")

Ce qu'on vient de faire porte un nom : **le traitement en flux** (*streaming*). C'est le geste de base du métier de data engineer, et il marche pour à peu près tout : compter, sommer, filtrer, agréger.

Sa limite : il ne marche pas pour ce qui a besoin de voir **toutes** les lignes en même temps (une médiane exacte, un tri global). Dans ce cas, on passe à un outil conçu pour ça — une base de données, Polars, ou Spark.

### Ramasse-miettes ou gestion explicite ?

Dernier point de cette partie, et c'est un vrai différenciateur entre les langages.

En **C ou C++**, tu réserves la mémoire à la main (`malloc`) et tu la rends à la main (`free`). Si tu oublies de rendre, ton programme grossit jusqu'à saturer la machine (une *fuite mémoire*). Si tu rends deux fois, il plante.

En **Python** (comme en Java ou en JavaScript), un **ramasse-miettes** (*garbage collector*) s'en charge : Python compte combien de noms pointent vers chaque objet, et dès que ce compteur tombe à zéro, il libère la mémoire tout seul. Tu n'as rien à faire.

Le prix à payer : ce comptage a un coût à chaque manipulation, et tu ne décides pas *quand* la mémoire est rendue. C'est une des raisons pour lesquelles Python est plus lent que C — et c'est le même marché que pour la vitesse : on paie un peu de performance pour beaucoup de tranquillité.

In [ ]:
import sys, gc

a = df                     # deux noms, un seul DataFrame en mémoire
b = df

print("nombre de noms qui pointent vers le DataFrame :", sys.getrefcount(df) - 1)

del a, b                   # `del` ne supprime pas les données : il supprime les *noms*
print("après del a, b                                :", sys.getrefcount(df) - 1)
print("le DataFrame est toujours là :", df.shape)

# Quand plus AUCUN nom ne pointe vers un objet, sa mémoire est rendue automatiquement.
# Le cas que le simple comptage ne sait PAS résoudre : deux objets qui se pointent l'un l'autre.
x = {}
y = {"vers_x": x}
x["vers_y"] = y
del x, y      # plus aucun nom vers eux... mais chacun est encore pointé par l'autre : compteurs à 1

print("\nramasse-miettes :", gc.collect(), "objet(s) nettoyé(s)")
print("C'est pour ces cas-là qu'il existe : il repère les groupes d'objets coupés du reste.")

## 3. Local ou cloud

Troisième question : **où est-ce que ça tourne ?** Il y a trois réponses possibles, et elles ne se valent pas selon ce que tu fais.

| | **Ma machine** | **Google Colab** | **Serveur cloud** |
|---|---|---|---|
| Mise en route | Installer Python, un éditeur, les bibliothèques : 1 à 2 heures la première fois | Ouvrir un onglet : 10 secondes | Créer un compte, une carte bancaire, configurer : une demi-journée |
| Puissance | Ce que tu as acheté. Rarement un GPU utilisable | 2 cœurs, ~12 Go de RAM, un **GPU T4 gratuit** quelques heures par jour | Ce que tu paies : 8, 64, 256 Go de RAM, plusieurs GPU |
| Persistance | Totale : tes fichiers restent | **Nulle** : la machine est effacée à la fermeture (ou après ~90 min d'inactivité) | Totale, tant que tu paies |
| Coût | 0 € en plus (tu as déjà l'ordinateur) | 0 € | À l'heure, et **ça tourne même quand tu dors** |
| Recommandé pour | Apprendre, coder au quotidien, données confidentielles | **Ce cours.** Essayer, partager un notebook, utiliser un GPU sans rien payer | Entraîner longtemps, données énormes, mettre en production |

Le piège du cloud, ce n'est pas le prix affiché : c'est la **machine oubliée allumée**. Un GPU à 2 €/h laissé tourner un week-end = 96 €. Tout le monde se fait avoir une fois.

Regardons d'abord ce qu'il y a sous le capot de la machine sur laquelle tu es en train de travailler.

In [ ]:
import platform, os

print("Python      :", platform.python_version())
print("Système     :", platform.system(), platform.machine())
print("Cœurs CPU   :", os.cpu_count())

try:
    import psutil
    print(f"RAM totale  : {psutil.virtual_memory().total / 1e9:.1f} Go")
except ImportError:
    print("RAM totale  : (psutil non installé)")

# Y a-t-il un GPU ? Sur Colab : Exécution → Modifier le type d'exécution → T4 GPU
gpu = os.popen("nvidia-smi --query-gpu=name,memory.total --format=csv,noheader 2>/dev/null").read().strip()
print("GPU         :", gpu if gpu else "aucun (CPU seulement) — c'est très bien pour ce cours")

### Ce que Colab te cache

Colab est agréable parce qu'il ment par omission : tout est déjà installé, donc on oublie qu'il y a quelque chose à installer.

Sauf que le jour où tu voudras faire tourner ton projet ailleurs — sur ta machine, chez un ami, sur un serveur — rien ne marchera, parce que rien n'y sera installé.

Regardons la liste de ce que Colab t'a offert sans le dire.

In [ ]:
!pip list | head -30

Plusieurs centaines de paquets. Ta machine, elle, n'en a aucun.

D'où le fichier le plus important d'un projet Python : **`requirements.txt`**. C'est la liste des bibliothèques nécessaires, avec leur version. Une seule commande, `pip install -r requirements.txt`, réinstalle exactement le même décor n'importe où.

Pourquoi épingler la version avec `==` ? Parce que pandas 2.3 et pandas 1.5 ne se comportent pas pareil. Sans version, ton code marche aujourd'hui et casse dans six mois, sur la machine de quelqu'un d'autre, sans que personne comprenne pourquoi.

La commande magique `%%writefile` écrit le contenu de la cellule dans un fichier, au lieu de l'exécuter.

In [ ]:
%%writefile requirements.txt
# Les bibliothèques de ce cours, avec leur version exacte.
# Sur une machine neuve : pip install -r requirements.txt
pandas==2.3.3
numpy==2.0.2
matplotlib==3.9.4
scikit-learn==1.6.1

In [ ]:
print(open("requirements.txt").read())

# Sur ta machine, dans un terminal, ces deux lignes suffisent à recréer le décor :
#   python -m venv .venv && source .venv/bin/activate
#   pip install -r requirements.txt
#
# Pour fabriquer ce fichier automatiquement à partir de ce qui est installé :
#   pip freeze > requirements.txt      (attention : il liste TOUT, y compris l'inutile)

### Les plateformes, et surtout leurs pages de tarifs

Quand on sort de Colab, on tombe sur deux familles d'offres :

**Louer des machines** (tu gères le modèle, l'entraînement, tout) :

- Amazon SageMaker — [présentation](https://aws.amazon.com/sagemaker/) · [tarifs](https://aws.amazon.com/sagemaker/pricing/)
- Google Vertex AI — [présentation](https://cloud.google.com/vertex-ai) · [tarifs](https://cloud.google.com/vertex-ai/pricing)

**Appeler un modèle par API** (tu ne gères rien, tu paies au token) :

- Console Anthropic (Claude) — [console](https://console.anthropic.com) · [tarifs](https://www.anthropic.com/pricing)
- Plateforme OpenAI (GPT) — [plateforme](https://platform.openai.com) · [tarifs](https://openai.com/api/pricing/)
- Google AI Studio (Gemini) — [studio](https://aistudio.google.com) · [tarifs](https://ai.google.dev/pricing)

Et pour Colab lui-même : [colab.research.google.com/signup](https://colab.research.google.com/signup) compare la version gratuite et Colab Pro.

Prends l'habitude d'ouvrir la page de tarifs **avant** de créer un compte, pas après la première facture. C'est le réflexe professionnel de base.

### À toi · Où lancer ce calcul ?

Écris `ou_lancer(taille_go, heures, besoin_gpu)` qui renvoie `"ma machine"`, `"Colab"` ou `"serveur cloud"`, selon ces règles :

1. si le calcul dure **plus de 8 heures**, ou si les données dépassent **10 Go** → `"serveur cloud"` (Colab coupe la session et n'a pas assez de RAM) ;
2. sinon, s'il faut un **GPU** → `"Colab"` (il est gratuit) ;
3. sinon → `"ma machine"`.

<details><summary>Indice</summary>

Trois `if`, dans cet ordre exact : le premier qui répond gagne. Attention à `or` dans la première condition.

</details>

In [ ]:
# À toi
def ou_lancer(taille_go, heures, besoin_gpu):
    return None

cas = [
    ("nettoyer un CSV de 2 Go",              2,  0.5, False),
    ("entraîner un réseau de neurones",      1,  2,   True),
    ("entraîner un gros modèle 3 jours",     4,  72,  True),
    ("analyser 40 Go de logs",              40,  3,   False),
]
for titre, go, h, gpu in cas:
    print(f"{ou_lancer(go, h, gpu)!s:15} ← {titre}")

<details><summary>Solution</summary>

```python
def ou_lancer(taille_go, heures, besoin_gpu):
    if heures > 8 or taille_go > 10:
        return "serveur cloud"
    if besoin_gpu:
        return "Colab"
    return "ma machine"
```

Résultat attendu :

```
ma machine      ← nettoyer un CSV de 2 Go
Colab           ← entraîner un réseau de neurones
serveur cloud   ← entraîner un gros modèle 3 jours
serveur cloud   ← analyser 40 Go de logs
```

L'ordre des `if` est ce qui compte : « long ou énorme » l'emporte sur « il me faut un GPU ». Un GPU gratuit ne sert à rien si la session est coupée au bout de 90 minutes.

</details>

## Ton livrable

Deux choses à produire avant de partir, pour **ton** projet à toi : celui de la séance 2, ton projet Kaggle, ou celui que tu as en tête.

**1. Le fichier de dépendances.** Il répond à une question simple : si je donne mon notebook à quelqu'un, qu'est-ce qu'il doit installer pour que ça marche ? Sans lui, un projet ne se relance nulle part. C'est la base de la **reproductibilité** : même code, mêmes versions, mêmes résultats.

**2. La fiche « où faire tourner mon projet ».** Où ça tourne, et pourquoi — une décision justifiée par un chiffre, pas une habitude.


In [ ]:
%%writefile requirements_mon_projet.txt
# Les bibliothèques de MON projet, avec leur version exacte.
# Pour connaître une version installée : import pandas ; print(pandas.__version__)
pandas==2.3.3
numpy==2.0.2
# ... à compléter : ajoute ce que ton projet importe vraiment, retire le reste


In [ ]:
# Ma fiche « où faire tourner mon projet » : remplis les six réponses, puis exécute.
fiche = {
    "mon projet":         "...",    # en une phrase
    "taille des données": "... Go",
    "durée estimée":      "... h",
    "besoin d'un GPU":    False,    # True ou False
    "je le lance sur":    "...",    # "ma machine", "Colab" ou "serveur cloud"
    "parce que":          "...",    # la raison, avec un chiffre
}

for cle, valeur in fiche.items():
    print(f"{cle:>20} : {valeur}")

reste = [c for c, v in fiche.items() if str(v).startswith("...")]
print("\n✅ Fiche complète, tu peux la recopier dans ton README." if not reste
      else "\n❌ Encore à remplir : " + ", ".join(reste))


## Bravo !

Tu as maintenant trois réflexes qui manquent à la plupart des gens qui débutent :

1. **La lenteur de Python n'est pas un problème**, parce qu'on ne s'en sert pas pour calculer : NumPy, pandas et Polars ont un moteur écrit en C ou en Rust. Tu l'as mesuré, sur 5 millions de valeurs. Si tu écris une boucle sur des données, cherche la version vectorisée.
2. **La mémoire se mesure**, avec `df.info(memory_usage="deep")`. Un `astype("category")` peut diviser la taille par 2, par 5, parfois par 8 ; et quand le fichier dépasse la RAM, on le lit par morceaux avec `chunksize` plutôt que d'acheter une plus grosse machine.
3. **Le lieu se décide avant, pas après**, et un projet se relance ailleurs grâce à son `requirements.txt`. Colab pour apprendre, le cloud quand c'est long ou énorme.

Rien de tout ça n'est à retenir par cœur. Ce qui est à retenir, c'est le geste : **avant de lancer, mesure.**
